Run the following cell only once: install nirfasterff and download the necessary sample data.
You may use the same code if you are running local. The correct version of nirfasterff should be automatically determined.

In [ ]:
import platform
import subprocess

version_string = platform.python_version()
major_ver = version_string.split('.')[1]
cpulib_name = 'cpu-linux-python3'+major_ver+'.zip'
gpulib_name = 'gpu-linux-python3'+major_ver+'.zip'
!git clone https://github.com/milabuob/nirfaster-FF.git
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/Conte69_on_TT_32k.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/IM_Gordon_2014_333_Parcels.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/headvol.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/headvol_optodes.mat

current_os = platform.system()
if current_os == "Darwin":
    cpulib_name = 'cpu-mac-python3'+major_ver+'.zip'
else:
    if current_os == "Windows":
        cpulib_name = 'cpu-win-python3'+major_ver+'.zip'
        gpulib_name = 'gpu-win-python3'+major_ver+'.zip'
        subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+gpulib_name])
        subprocess.run(['unzip', gpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
        subprocess.run(['rm', gpulib_name])
    elif current_os == "Linux":
        cpulib_name = 'cpu-linux-python3'+major_ver+'.zip'
        gpulib_name = 'gpu-linux-python3'+major_ver+'.zip'
        subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+gpulib_name])
        subprocess.run(['unzip', gpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
        subprocess.run(['rm', gpulib_name])
        
subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+cpulib_name])
subprocess.run(['unzip', cpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
subprocess.run(['rm', cpulib_name])

!pip install ipympl
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
import sys
sys.path.insert(1, 'nirfaster-FF/')
import nirfasterff as ff # ff is short for fast and furious
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import copy
import math
# enable interactive matplotlib
# comment out the following lines if you run local
from google.colab import output
output.enable_custom_widget_manager()
%matplotlib ipympl

In [ ]:
# define a simple system of a single source and 4 detectors in a semi-infinite model and generate data
# Source-Detector Distance
SDS = np.arange(10,41,10)

# from this paper, we have tissue specific HbO, Hb and scattering properties: https://doi.org/10.1117/1.NPh.8.1.015013
# Region    	HbO (mM)	Hb (mM)	Sa (mm−1)	Sp
# Skin	        0.0575	    0.0313	0.53	    1.15
# Skull	        0.0443	    0.0195	0.72	    0.89
# CSF	        0.0110	    0.0083	0.30	    0
# Gray matter	0.0559	    0.0350	0.50	    1.73
# White matter	0.0680	    0.0265	0.81	    1.31
# Let's assume a homogeneous model, based on Gray Matter
# let's define a chromophore matrix
C = np.array([[0.0559,    0.0350]])

# assuming two wavelength of 760 and 850 nm, from NIRFAST we gave these extinction coeff
# wavelenth(nm) HbO(mm^-1/mM)   HbR(mm^-1/mM)
# 760           1.3495580e-01   3.5662416e-01
# 850           2.4365740e-01   1.5921100e-01
# let's define an extinction coefficient matrix
E = np.array([[1.3495580e-01,   3.5662416e-01],
              [2.4365740e-01,   1.5921100e-01]])

# we can now calculate mua at each wavelength using Beer's law
mua = np.dot(C,E)

# we can calulate musp using Mie's theory
Sa = np.array([0.5])
Sp = np.array([1.73])
musp = np.zeros((1,2))
wv = np.array([760, 850])
musp[:,0] = Sa * (wv[0] / 1000) ** (-Sp)
musp[:,1] = Sa * (wv[1] / 1000) ** (-Sp)

print(f"mua(mm-1) = 760 nm; 850 nm\n{mua}")
print(f"musp(mm-1) = 760 nm; 850 nm\n{musp}")

In [ ]:
# define HRF function
from scipy.special import gamma
def canonicalHRF(duration, Fs, peakTime=6., uShootTime=16., peakDisp=1.,
                 uShootDisp=1., ratio=1/6):
    # canonical hrf function adapted from nirs-toolbox, using standard SPM parameters
    t = np.arange(0, duration, 1/Fs)
    a1 = peakTime
    a2 = uShootTime
    b1 = peakDisp
    b2 = uShootDisp
    c  = ratio
    h = (b1**a1) * t**(a1-1)*np.exp(-b1*t)/gamma(a1) - c*(b2**a2)*(t**(a2-1))*np.exp(-b2*t)/gamma(a2)
    return h/h.max()

In [ ]:
# create a Haemodynamic Response Function (HRF) for HbO and Hb
hrf_length = 30 # duration (sec)
Fs = 10.0 # sampling rate (hz)
hrf_HbO = canonicalHRF(hrf_length, Fs)

# assuming Hb HRF is -0.25 of HbO
hrf_Hb = -0.25 * hrf_HbO
tvec = np.arange(0, hrf_length, 1/Fs)

plt.close('all')
plt.plot(tvec, hrf_HbO, 'r', label='HbO')
plt.plot(tvec, hrf_Hb, label='Hb')
plt.legend()
plt.show()

In [ ]:
# Calculate change in mua from the HRF using Beer's Law, and use a fraction (i.e. not a high mua change)
hrf = np.vstack((hrf_HbO,hrf_Hb))
delta_mua = np.dot(E,hrf)  # * 0.005

plt.close('all')
plt.plot(tvec, delta_mua[0,:], 'r', label='760nm')
plt.plot(tvec, delta_mua[1,:], label='850nm')
plt.xlabel('time (s)')
plt.ylabel(r'$\delta \mu_a$')
plt.legend()
plt.show()

In [ ]:
# Generate data, from these Channels using semi-infinite model
data_760nm = np.zeros((delta_mua.shape[1], len(SDS)))
data_850nm = np.zeros((delta_mua.shape[1], len(SDS)))

for t in range(delta_mua.shape[1]):
    data_760nm[t,:] = ff.forward.semi_infinite_CW(mua[0,0]+delta_mua[0,t],musp[0,0],1.33,SDS,z=0)
    data_850nm[t,:] = ff.forward.semi_infinite_CW(mua[0,1]+delta_mua[1,t],musp[0,1],1.33,SDS,z=0)

plt.close('all')
plt.subplot(2,1,1)
plt.semilogy(tvec, data_760nm)
plt.title('760 nm')
plt.subplot(2,1,2)
plt.semilogy(tvec, data_850nm)
plt.title('850 nm')
plt.tight_layout()
plt.show()

In [ ]:
# calculate OD, using 1st frame as baseline
dOD_760nm = -np.log(data_760nm / data_760nm[0,:])
dOD_850nm = -np.log(data_850nm / data_850nm[0,:])

plt.close('all')
plt.subplot(2,1,1)
plt.plot(tvec, dOD_760nm)
plt.title('760 nm')
plt.subplot(2,1,2)
plt.plot(tvec, dOD_850nm)
plt.title('850 nm')
plt.tight_layout()
plt.show()

Now let's add some realistic noise to the data. Noise model from doi: 10.1117/1.NPh.6.3.035007. Original paper used 690nm and 830nm, but let's largely assume the numbers can be applied to our case as well.

Percentage noise of OD is $ae^{br}+ce^{dr}$, where r is source-detector distance, a, b, c, and d are coefficients, which were found to be:
Coefficients | 830 nm | 690 nm
-------------| -------| ------
a            | 0.6019 | 0.2502
b            | 0.01052| 0.02913
c            | 9.685e-5| 4.625e-6
d            | 0.1382 | 0.2128

In [ ]:
noise_percent850 = 0.6019*np.exp(0.01052*SDS) + 9.685e-5*np.exp(0.1382*SDS)
noise_percent760 = 0.2502*np.exp(0.02913*SDS) + 4.625e-6*np.exp(0.2128*SDS)

noise850 = np.zeros(data_850nm.shape)
noise760 = np.zeros(data_760nm.shape)
for i in range(len(SDS)):
    noise850[:,i] = np.random.randn(delta_mua.shape[1]) * (np.log(data_850nm[0,i]) * noise_percent850[i]/100)
    noise760[:,i] = np.random.randn(delta_mua.shape[1]) * (np.log(data_760nm[0,i]) * noise_percent760[i]/100)

OD_850nm_noisy = np.log(data_850nm) + noise850
OD_760nm_noisy = np.log(data_760nm) + noise760

plt.close('all')
plt.subplot(2,1,1)
plt.plot(tvec, OD_760nm_noisy)
plt.title('760 nm')
plt.subplot(2,1,2)
plt.plot(tvec, OD_850nm_noisy)
plt.title('850 nm')
plt.tight_layout()
plt.show()

And now calculate dOD

In [ ]:
# calculate OD, using 1st frame as baseline
# note that we have already taken the logarithm, so the previously used division is now subtraction
dOD_760nm_noisy = -(OD_760nm_noisy - OD_760nm_noisy[0,:])
dOD_850nm_noisy = -(OD_850nm_noisy - OD_850nm_noisy[0,:])

plt.close('all')
plt.subplot(2,1,1)
plt.plot(tvec, dOD_760nm_noisy)
plt.title('760 nm')
plt.subplot(2,1,2)
plt.plot(tvec, dOD_850nm_noisy)
plt.title('850 nm')
plt.tight_layout()
plt.show()

Apply Modified Beer-Lambert law: $\Delta OD=\pmatrix{\xi_{\lambda1,HbO}DPF_{lambda1}r & \xi_{\lambda1,Hb}DPF_{lambda1}r)\\ \xi_{\lambda2,HbO}DPF_{lambda2}r & \xi_{\lambda2,Hb}DPF_{lambda2}r} \pmatrix{\Delta HbO\\ \Delta Hb}$
*Think:* classically DPF is different per wavelength, but it has become common to use the same value for both wavelenths in recent years. How much difference does it make?

In [ ]:
# let's apply the Modified Beer-Lambert solution here for a channel with highest OD
dmua = np.zeros((2,300))
DPF = 6
channel = 0

dmua[0,:] = dOD_760nm[:,channel] / (DPF * SDS[channel])
dmua[1,:] = dOD_850nm[:,channel] / (DPF * SDS[channel])
result = np.linalg.inv(E) @ dmua

# for plotting purpose, normalise and compare to original
hrf_recoeverd = result / np.max(np.abs(result))

plt.close('all')
plt.plot(tvec, hrf_recoeverd[0,:], 'r', label='HbO')
plt.plot(tvec, hrf_recoeverd[1,:], 'b', label='Hb')
plt.plot(tvec, hrf_HbO, 'rx', label='HbO: target')
plt.plot(tvec, hrf_Hb, 'bx', label='Hb')
plt.legend()
plt.show()

In [ ]:
# Let's repeat with noisy data
dmua = np.zeros((2,300))
DPF = 6
channel = 0

dmua[0,:] = dOD_760nm_noisy[:,channel] / (DPF * SDS[channel])
dmua[1,:] = dOD_850nm_noisy[:,channel] / (DPF * SDS[channel])
result = np.linalg.inv(E) @ dmua

# for plotting purpose, normalise and compare to original
hrf_recoeverd = result / np.max(np.abs(result))

plt.close('all')
plt.plot(tvec, hrf_recoeverd[0,:], 'r', label='HbO')
plt.plot(tvec, hrf_recoeverd[1,:], 'b', label='Hb')
plt.plot(tvec, hrf_HbO, 'rx', label='HbO: target')
plt.plot(tvec, hrf_Hb, 'bx', label='Hb')
plt.legend()
plt.show()